# Hugging Face Model Exploration & FastAPI Demonstration
**Course**: Computational Thinking (Tư duy tính toán)  
**Faculty**: Information Technology, University of Science, VNU-HCM  
**Lab**: Practice 01 - Application Programming Interface (Web API)  

---
## 1. Introduction
This notebook demonstrates:
1. Loading and testing the Hugging Face model (`distilbert/distilbert-base-uncased-finetuned-sst-2-english`) directly using `transformers`.
2. Interacting with the deployed FastAPI Web API using Python's `requests` library.
3. Validating normal test cases (positive/negative sentiment) and error cases (empty text, invalid payloads).

## 2. Direct Model Exploration with Hugging Face Pipeline
Before building and calling the API, let's explore how the `distilbert-base-uncased-finetuned-sst-2-english` model works directly via the Transformers `pipeline` interface.

In [ ]:
import torch
from transformers import pipeline

# Check device availability (CUDA GPU or CPU)
device = 0 if torch.cuda.is_available() else -1
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

# Load the sentiment analysis pipeline
model_name = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
classifier = pipeline("sentiment-analysis", model=model_name, device=device)
print(f"Successfully loaded pipeline for model: {model_name}")

In [ ]:
# Sample test inputs
sample_sentences = [
    "This machine learning lab is exceptionally well-structured and exciting!",
    "The download speed was terrible and the installation failed multiple times.",
    "The meeting was scheduled at 3 PM."
]

for sentence in sample_sentences:
    result = classifier(sentence)[0]
    print(f"Input : {sentence}")
    print(f"Result: Label = {result['label']}, Confidence = {result['score']:.4f}")
    print("-" * 60)

## 3. Testing the FastAPI Web API via HTTP Requests
Once the FastAPI server is running (e.g. `uvicorn app.main:app --reload`), we can consume its endpoints via HTTP using Python's `requests` library.

In [ ]:
import json
import requests

BASE_URL = "http://127.0.0.1:8000"

# 1. Test Root Endpoint (GET /)
res_root = requests.get(f"{BASE_URL}/")
print(f"GET / -> Status {res_root.status_code}")
print(json.dumps(res_root.json(), indent=2))

In [ ]:
# 2. Test Health Endpoint (GET /health)
res_health = requests.get(f"{BASE_URL}/health")
print(f"GET /health -> Status {res_health.status_code}")
print(json.dumps(res_health.json(), indent=2))

In [ ]:
# 3. Test POST /predict with a single sentence
payload_single = {
    "text": "Developing Web APIs with FastAPI and Hugging Face is an incredible developer experience!"
}

res_predict = requests.post(f"{BASE_URL}/predict", json=payload_single)
print(f"POST /predict -> Status {res_predict.status_code}")
print(json.dumps(res_predict.json(), indent=2))

In [ ]:
# 4. Test POST /predict with batch of inputs
payload_batch = {
    "texts": [
        "The artificial intelligence curriculum at HCMUS is comprehensive and inspiring.",
        "I ran out of disk space and the system crashed without saving my work."
    ]
}

res_batch = requests.post(f"{BASE_URL}/predict", json=payload_batch)
print(f"POST /predict (Batch) -> Status {res_batch.status_code}")
print(json.dumps(res_batch.json(), indent=2))

In [ ]:
# 5. Test Error Handling (Empty Payload & Blank Text)
bad_payload = {"text": "    "}
res_err = requests.post(f"{BASE_URL}/predict", json=bad_payload)
print(f"POST /predict (Invalid) -> Status {res_err.status_code}")
print(json.dumps(res_err.json(), indent=2))

## 4. Conclusion
- The Hugging Face model accurately predicts sentiment classification.
- The FastAPI application provides clean, well-structured REST endpoints (`/`, `/health`, `/predict`).
- Automatic validation handles invalid and missing inputs with informative HTTP error responses.